In [40]:
import pandas as pd
import numpy as np
import warnings
import seaborn as sns 
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression
from sklearn import metrics
from scipy.stats import skew, kurtosis
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.sandbox.stats.runs import runstest_1samp
from scipy import stats
warnings.filterwarnings('ignore')

In [41]:
sber = pd.read_csv(r'C:\Users\user\Desktop\AlgoTrading\data\LKOH.txt')

In [42]:
# Преобразуем "сырой" датафрейм
def good_dataframe(data, timeframe):
  """Преобразует сырые рыночные данные в чистый DataFrame с правильными типами и индексом времени
    
    Подготавливает данные для технического анализа.
    
    Args:
        data (pd.DataFrame): Исходный DataFrame с рыночными данными, содержащий столбцы:
            ['<TICKER>', '<PER>', '<DATE>', '<TIME>', '<OPEN>', '<HIGH>', '<LOW>', '<CLOSE>', '<VOL>']
            
    Returns:
        tuple: Возвращает кортеж из двух DataFrame:
            - Основной DataFrame
            - Копия DataFrame для безопасного резервирования
            
    Processing Logic:
        1. Удаление избыточных столбцов
        2. Переименование столбцов в human-friendly формат
        3. Преобразование типов данных
        4. Создание правильного временного индекса
    
    """
  # 1. Делаем копию, чтобы не изменялся исходный датафрейм
  data = data.copy()
  
  # 2. Переименовываем столбцы для удобства работы
  data.columns = ['ticker', 'per', 'date', 'time', 'open', 'high', 'low', 'close', 'volume']
    
  # 3. Преобразуем дату из формата YYYYMMDD в datetime
  data['date'] = pd.to_datetime(data['date'], format='%Y%m%d')
    
  # 4. Обрабатываем время (HHMMSS -> datetime.time)
  data['time'] = pd.to_datetime(data['time'], format='%H%M%S').dt.time
    
  # 5. Комбинируем дату и время в единую метку времени
  data['time'] = pd.to_datetime(
        data['date'].astype('str') + ' ' + data['time'].astype('str'))
    
  # 6. Удаляем отдельный столбец даты (теперь он в индексе)
  data.drop(['date'], inplace=True, axis=1)
  
  # 7. Установка индекса
  data_final = data.set_index('time')
  
  
  
  def new_timeframe(data, timeframe):
    """Преобразует минутные данные (1М) в указанный временной интервал, сохраняя структуру OHLCV-данных.
    
    Использует принципы агрегации свечных данных:
    - Open - первое значение периода
    - High - максимум периода
    - Low - минимум периода
    - Close - последнее значение периода
    - Volume - сумма объема за период

    Args:
        data (pd.DataFrame): Исходный DataFrame с 1-минутными данными, 
                            должен содержать колонки ['open', 'high', 'low', 'close', 'volume']
                            и иметь DateTimeIndex
        timeframe (str): Желаемый таймфрейм из списка доступных:
                        ['5 min', '15 min', '30 min', '1h', '2h', '4h', 'D']

    Returns:
        pd.DataFrame: Новый DataFrame с преобразованными данными в указанном таймфрейме
        
    Raises:
        ValueError: Если передан неподдерживаемый timeframe
    """

    dict_tf = {'5 min' : '5min', '15 min' : '15min', '30 min' : '30min',
               '1h' : '1h', '2h' : '2h', '4h' : '4h', 'D' : 'D'}

    return_data = data.resample(dict_tf[timeframe]).agg({
            'ticker': 'first',
            'per': 'first',
            'open': 'first',
            'high': 'max',
            'low': 'min',
            'close': 'last',
            'volume': 'sum'
        }).dropna()
    

    return_data['per'] = timeframe
    return return_data
  
  result = new_timeframe(data_final, timeframe)
  
  result = result.reset_index()
  
  return result

In [43]:
sber_5 = good_dataframe(sber, '5 min')
sber_5

,time,ticker,per,open,high,low,close,volume
0,2009-01-11 10:30:00,LKOH,5 min,9851.0,10099.0,9750.0,10000.0,631
1,2009-01-11 10:35:00,LKOH,5 min,10007.0,10035.0,9921.0,9966.0,663
2,2009-01-11 10:40:00,LKOH,5 min,9994.0,10017.0,9946.0,9989.0,606
3,2009-01-11 10:45:00,LKOH,5 min,9987.0,10050.0,9962.0,10017.0,289
4,2009-01-11 10:50:00,LKOH,5 min,10000.0,10065.0,10000.0,10044.0,251
...,...,...,...,...,...,...,...,...
638205,2025-06-30 23:25:00,LKOH,5 min,65957.0,65957.0,65957.0,65957.0,4
638206,2025-06-30 23:30:00,LKOH,5 min,65953.0,65953.0,65866.0,65953.0,18
638207,2025-06-30 23:35:00,LKOH,5 min,65953.0,65980.0,65953.0,65980.0,13
638208,2025-06-30 23:40:00,LKOH,5 min,65958.0,65990.0,65958.0,65976.0,15


In [44]:
sber_5.info()

<class 'pandas.DataFrame'>
RangeIndex: 638210 entries, 0 to 638209
Data columns (total 8 columns):
 #   Column  Non-Null Count   Dtype         
---  ------  --------------   -----         
 0   time    638210 non-null  datetime64[us]
 1   ticker  638210 non-null  str           
 2   per     638210 non-null  str           
 3   open    638210 non-null  float64       
 4   high    638210 non-null  float64       
 5   low     638210 non-null  float64       
 6   close   638210 non-null  float64       
 7   volume  638210 non-null  int64         
dtypes: datetime64[us](1), float64(4), int64(1), str(2)
memory usage: 44.6 MB


In [45]:
# Преобразуем датафрейм для удобства работы с 2 свечными паттернами
def shift_features_2_candle(data):
    """Смещает все основные столбцы на 1 период назад

    Args:
        data (pd.DataFrame): Исходный DataFrame с рыночными данными, содержащий столбцы:
            ['time', 'ticker', 'per', 'open', 'high', 'low', 'close', 'volume']

    Returns:
        data (pd.DataFrame): Новый DataFrame с преобразованными данными, содержащий столбцы:
        ['ticker', 'per', 'open_N', 'open_N-1', 'close_N', 'close_N-1', 'low_N',
       'low_N-1', 'high_N', 'high_N-1', 'volume_N', 'volume_N-1', 'time_N',
       'time_N-1']
    """
    data_c = data.copy()
    for i in ['open', 'close', 'low', 'high', 'volume', 'time']:
        data_c[f"{i}_N"] = data[i]
        data_c[f'{i}_N-1'] = data[i].shift(1)
    data_c.drop(['open', 'close', 'low', 'high', 'volume', 'time'], axis=1, inplace=True)
    data_c.dropna(inplace=True)
    return data_c

sber_5s = shift_features_2_candle(sber_5)
sber_5s.head(3)

,ticker,per,open_N,open_N-1,close_N,close_N-1,low_N,low_N-1,high_N,high_N-1,volume_N,volume_N-1,time_N,time_N-1
1,LKOH,5 min,10007.0,9851.0,9966.0,10000.0,9921.0,9750.0,10035.0,10099.0,663,631.0,2009-01-11 10:35:00,2009-01-11 10:30:00
2,LKOH,5 min,9994.0,10007.0,9989.0,9966.0,9946.0,9921.0,10017.0,10035.0,606,663.0,2009-01-11 10:40:00,2009-01-11 10:35:00
3,LKOH,5 min,9987.0,9994.0,10017.0,9989.0,9962.0,9946.0,10050.0,10017.0,289,606.0,2009-01-11 10:45:00,2009-01-11 10:40:00


In [46]:
def bullish_counterattack(data):
    """
    Свечная модель, состоящая из 2 свечей. Первая свеча падающая, вторая свеча растущая.
    Цены закрытия равны или очень близки друг к другу.
    
    Args:
        data (pd.DataFrame): Исходный DataFrame с рыночными данными, содержащий столбцы:
        ['ticker', 'per', 'open_N', 'open_N-1', 'close_N', 'close_N-1', 'low_N',
       'low_N-1', 'high_N', 'high_N-1', 'volume_N', 'volume_N-1', 'time_N',
       'time_N-1']

    Returns:
        data (pd.DataFrame): Исходный DataFrame вместе с дополнительными 3 столбцами:
        - pattern : 1 - 2 свечи наблюдения образуют паттерн, 0 - паттетна нет.
        - signal : 1 - на предыдущей свече был паттерн, 0 - паттерна не было, сигнала на покупку на данной свече нет
        - strategy : Название стратегии - 'bullish_counterattack'
        
    """
    data = data.copy()
    data['pattern'] = 0
    data['signal'] = 0
    data['strategy'] = 'bullish_counterattack'
    
    # Векторизованные вычисления
    close_N = data['close_N']
    close_N_1 = data['close_N-1']
    
    body_N = data['close_N'] - data['open_N']
    body_N_1 = data['close_N-1'] - data['open_N-1']
    

    # Базовое условие для просвета в облаках
    base_condition = (
        (body_N_1 < 0) & 
        (body_N > 0) &
        ((np.abs(close_N_1 - close_N) / close_N) * 100 <= 0.05))
    
    # Отмечаем 2 свечи паттерна
    pattern_mask = base_condition
    data.loc[pattern_mask, 'pattern'] = 1
    # Сигнал - следующая свеча после завершения паттерна
    data.loc[pattern_mask.shift(1).fillna(False), 'signal'] = 1
        
    
    return data
sber_good = bullish_counterattack(sber_5s)

In [47]:
sber_good[sber_good['pattern'] == 1]

,ticker,per,open_N,open_N-1,close_N,close_N-1,low_N,low_N-1,high_N,high_N-1,volume_N,volume_N-1,time_N,time_N-1,pattern,signal,strategy
28,LKOH,5 min,10095.0,10103.0,10103.0,10098.0,10086.0,10091.0,10122.0,10106.0,185,217.0,2009-01-11 12:50:00,2009-01-11 12:45:00,1,0,bullish_counterattack
37,LKOH,5 min,10056.0,10066.0,10065.0,10065.0,10049.0,10065.0,10065.0,10099.0,341,13.0,2009-01-11 13:35:00,2009-01-11 13:30:00,1,0,bullish_counterattack
51,LKOH,5 min,10137.0,10138.0,10138.0,10137.0,10135.0,10127.0,10141.0,10143.0,25,53.0,2009-01-11 14:45:00,2009-01-11 14:40:00,1,0,bullish_counterattack
80,LKOH,5 min,10079.0,10086.0,10085.0,10082.0,10050.0,10082.0,10085.0,10095.0,79,9.0,2009-01-11 17:15:00,2009-01-11 17:10:00,1,0,bullish_counterattack
93,LKOH,5 min,10140.0,10145.0,10143.0,10140.0,10137.0,10133.0,10148.0,10145.0,74,134.0,2009-01-11 18:35:00,2009-01-11 18:30:00,1,0,bullish_counterattack
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
638130,LKOH,5 min,65940.0,65952.0,65955.0,65941.0,65907.0,65919.0,65955.0,65952.0,52,23.0,2025-06-30 15:40:00,2025-06-30 15:35:00,1,0,bullish_counterattack
638145,LKOH,5 min,66060.0,66111.0,66083.0,66060.0,66024.0,66060.0,66083.0,66124.0,94,68.0,2025-06-30 16:55:00,2025-06-30 16:50:00,1,0,bullish_counterattack
638174,LKOH,5 min,66081.0,66096.0,66102.0,66089.0,66081.0,65966.0,66102.0,66123.0,14,78.0,2025-06-30 20:00:00,2025-06-30 19:35:00,1,0,bullish_counterattack
638193,LKOH,5 min,66077.0,66125.0,66106.0,66077.0,66027.0,66047.0,66106.0,66125.0,11,34.0,2025-06-30 21:55:00,2025-06-30 21:50:00,1,0,bullish_counterattack
